## ✈️ CASE 4: Sentiment Analysis of Passenger Feedback 
**Objective:**
Analyze how travelers express opinions about airlines on Twitter (positive, neutral, negative).
Use NLP techniques to classify tweets and find which airlines have the most positive or negative sentiment.

### Step 1: Load & Inspect Data

In [0]:
df = spark.table("workspace.default.4_twitter_us_airline_sentiment")

df.printSchema()
display(df.limit(5))


root
 |-- tweet_id: double (nullable = true)
 |-- airline_sentiment: string (nullable = true)
 |-- airline_sentiment_confidence: double (nullable = true)
 |-- negativereason: string (nullable = true)
 |-- negativereason_confidence: double (nullable = true)
 |-- airline: string (nullable = true)
 |-- airline_sentiment_gold: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negativereason_gold: string (nullable = true)
 |-- retweet_count: long (nullable = true)
 |-- text: string (nullable = true)
 |-- tweet_coord: string (nullable = true)
 |-- tweet_created: string (nullable = true)
 |-- tweet_location: string (nullable = true)
 |-- user_timezone: string (nullable = true)



tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
5.69944E17,negative,1.0,Flight Attendant Complaints,1.0,US Airways,null,thomashoward88,null,0,"@USAirways US 728 pilot started the flight by announcing arrival into Gatwick, not Heathrow. Moments of concern for all aboard.",null,23-02-2015 11:37,null,null
5.69944E17,negative,1.0,Customer Service Issue,0.6772,US Airways,null,thomashoward88,null,0,@USAirways US 728 stated their issues as: no one around to close said door. More strange. Now we're not hearing from pilot for long periods.,null,23-02-2015 11:36,null,null
5.69944E17,negative,1.0,Bad Flight,0.684,US Airways,null,ac_choplick,null,0,@USAirways Seat 8D on Flight 545 last night is the worst seat I've ever been in. No leg room! #dividendsmember http://t.co/NUhpLNXRIq,null,23-02-2015 11:35,null,Hawaii
5.69943E17,negative,1.0,Flight Attendant Complaints,0.6723,US Airways,null,Blasterjaxx,null,1,@USAirways One of your workers was very rude with us. We can tell you in DM cuz it won't be any good publicity...,null,23-02-2015 11:34,TotalWorldJaxxination,null
5.69943E17,negative,0.652,Late Flight,0.342,US Airways,null,thomashoward88,null,0,"@USAirways US 728 stated their issues as: plane not moving as cargo door open on plane. Umm, ok. A little strange.",null,23-02-2015 11:34,null,null


### Step 2: Data Cleaning & Standardization

In [0]:
from pyspark.sql.functions import trim, col

# Select only the relevant columns
df = df.select("tweet_id", "airline", "text", "airline_sentiment")

# Clean spaces and remove missing values
df = df.filter(col("text").isNotNull() & col("airline_sentiment").isNotNull())
df = df.select([trim(col(c)).alias(c) for c in df.columns])

print("✅ Cleaned columns:")
print(df.columns)
display(df.limit(5))


✅ Cleaned columns:
['tweet_id', 'airline', 'text', 'airline_sentiment']


tweet_id,airline,text,airline_sentiment
5.69944E17,US Airways,"@USAirways US 728 pilot started the flight by announcing arrival into Gatwick, not Heathrow. Moments of concern for all aboard.",negative
5.69944E17,US Airways,@USAirways US 728 stated their issues as: no one around to close said door. More strange. Now we're not hearing from pilot for long periods.,negative
5.69944E17,US Airways,@USAirways Seat 8D on Flight 545 last night is the worst seat I've ever been in. No leg room! #dividendsmember http://t.co/NUhpLNXRIq,negative
5.69943E17,US Airways,@USAirways One of your workers was very rude with us. We can tell you in DM cuz it won't be any good publicity...,negative
5.69943E17,US Airways,"@USAirways US 728 stated their issues as: plane not moving as cargo door open on plane. Umm, ok. A little strange.",negative


### Step 3: SQL EDA — Sentiment Distribution
View how many tweets per airline were positive, neutral, or negative.

In [0]:
df.createOrReplaceTempView("tweets")

# Sentiment counts per airline
spark.sql("""
SELECT airline, 
       COUNT(CASE WHEN airline_sentiment = 'positive' THEN 1 END) AS positive,
       COUNT(CASE WHEN airline_sentiment = 'neutral' THEN 1 END) AS neutral,
       COUNT(CASE WHEN airline_sentiment = 'negative' THEN 1 END) AS negative,
       COUNT(*) AS total
FROM tweets
GROUP BY airline
ORDER BY total DESC
""").show()


+----------+--------+-------+--------+-----+
|   airline|positive|neutral|negative|total|
+----------+--------+-------+--------+-----+
|US Airways|     264|    356|    2365| 2985|
|  American|     336|    463|    1958| 2757|
|     Delta|       0|      0|       1|    1|
+----------+--------+-------+--------+-----+



### Step 4: Text Preprocessing for ML
Convert tweet text into numerical vectors suitable for machine learning.

In [0]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml import Pipeline

# Encode labels (positive=2, neutral=1, negative=0)
indexer = StringIndexer(inputCol="airline_sentiment", outputCol="label")

# Tokenize tweets
tokenizer = Tokenizer(inputCol="text", outputCol="words")

# Remove stopwords
remover = StopWordsRemover(inputCol="words", outputCol="filtered")

# Convert to term frequency features
tf = HashingTF(inputCol="filtered", outputCol="rawFeatures", numFeatures=10000)

# Compute TF-IDF
idf = IDF(inputCol="rawFeatures", outputCol="features")

print("✅ Preprocessing pipeline ready")


✅ Preprocessing pipeline ready


### Step 5: Model Training (Logistic Regression)
Train a sentiment classification model to predict tweet polarity (positive, neutral, negative).

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
pipeline = Pipeline(stages=[indexer, tokenizer, remover, tf, idf, lr])

# Split dataset
train, test = df.randomSplit([0.8, 0.2], seed=42)

# Train model
model = pipeline.fit(train)

# Predictions
pred = model.transform(test)

display(pred.select("text", "airline_sentiment", "prediction").limit(10))


text,airline_sentiment,prediction
. @USAirways It's been Cancelled Flighted. Your SM response is slow.,negative,0.0
@USAirways can't even get on hold to wait to speak to someone-awesome,negative,0.0
@USAirways it takes a month?,negative,1.0
@USAirways well its 11:45am and just got an email that my 11am flight is delayed-thats not right,negative,0.0
@USAirways I did and it's been a disaster. You had me sitting on the runway only to bring the plane back to the gate smh,negative,0.0
@USAirways I expect something more than telling me to see an agent to rebook my flight...,negative,0.0
@USAirways I got up at 2 am for a 5 am flight from bos to Charlotte which I found was Cancelled Flightled once I got to the gate (1),negative,0.0
"@USAirways on your website and on your boards at Logan it said it was on time, so we went through security and got to the gate (2)",neutral,0.0
@USAirways and it still says it's on time on your website btw,negative,1.0
@USAirways thanks,positive,2.0


### Step 6: Model Evaluation

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(pred)
print("✅ Model Accuracy:", round(accuracy, 3))


✅ Model Accuracy: 0.765


### Step 7: SQL Visualization: Sentiment by Airline
Aggregate predicted sentiments by airline to visualize brand perception.

In [0]:
pred.createOrReplaceTempView("sentiment_predictions")

spark.sql("""
SELECT airline,
       ROUND(AVG(CASE WHEN prediction = 2 THEN 1 ELSE 0 END)*100,2) AS pct_positive,
       ROUND(AVG(CASE WHEN prediction = 1 THEN 1 ELSE 0 END)*100,2) AS pct_neutral,
       ROUND(AVG(CASE WHEN prediction = 0 THEN 1 ELSE 0 END)*100,2) AS pct_negative
FROM sentiment_predictions
GROUP BY airline
ORDER BY pct_positive DESC
""").show()


+----------+------------+-----------+------------+
|   airline|pct_positive|pct_neutral|pct_negative|
+----------+------------+-----------+------------+
|  American|       14.05|      18.92|       67.03|
|US Airways|        8.78|      16.55|       74.66|
|     Delta|         0.0|        0.0|       100.0|
+----------+------------+-----------+------------+



### Step 8: Visualization

In [0]:
%sql
SELECT airline,
       ROUND(AVG(CASE WHEN prediction = 2 THEN 1 ELSE 0 END)*100,2) AS Positive,
       ROUND(AVG(CASE WHEN prediction = 1 THEN 1 ELSE 0 END)*100,2) AS Neutral,
       ROUND(AVG(CASE WHEN prediction = 0 THEN 1 ELSE 0 END)*100,2) AS Negative
FROM sentiment_predictions
GROUP BY airline;


airline,Positive,Neutral,Negative
American,14.05,18.92,67.03
US Airways,8.78,16.55,74.66
Delta,0.0,0.0,100.0


Databricks visualization. Run in Databricks to view.